# 05 — Medal Forecasting Model
**Project:** LA 2028 Olympic Games Strategic Analysis  
**Analyst:** Shabeeb | SportsFanatics Consulting  

## Model Overview

### Objective
Forecast total medal counts per NOC at the LA 2028 Summer Olympics.

### Approach
We train a **rolling-window regression model** on country-level features aggregated per Games edition, then project forward to 2028. We train two models — **Random Forest** and **Gradient Boosting** — and ensemble their predictions.

### Features Used
| Feature | Description |
|---|---|
| `prev_medals` | Medals won at the previous Games |
| `prev2_medals` | Medals won 2 Games ago |
| `avg_medals_3` | 3-Games rolling average |
| `athletes_sent` | Number of athletes entered |
| `events_entered` | Number of events entered |
| `is_host` | Whether the NOC is the host nation |
| `medal_rate_3` | Medal rate over last 3 Games |
| `sport_breadth` | Number of sports entered |

### Assumptions & Limitations
- Model trained only on 1988–2016 (post-boycott era) for comparability
- GDP not included (not in dataset) — captured indirectly by historical medal performance
- Geopolitical disruptions (Russia/Belarus exclusion) applied as post-model adjustments
- Predictions are probabilistic ranges, not point forecasts
- New sports (Flag Football, Cricket, etc.) contribute additional medals not captured by historical features — addressed via an addendum


## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import logging
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, LeaveOneGroupOut
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import sklearn

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

BASE_DIR   = Path('..')
RAW_DATA   = BASE_DIR / 'data' / 'raw' / 'athlete_events.csv'
OUT_TABLES = BASE_DIR / 'outputs' / 'tables'
OUT_FIGS   = BASE_DIR / 'outputs' / 'figures'

OLY = dict(blue='#0085C7', yellow='#F4C300', black='#000000',
           green='#009F6B', red='#DF0024', bg='#F8F9FA')

RANDOM_STATE = 42
print(f'scikit-learn version: {sklearn.__version__}')
logger.info('Setup complete.')

INFO: Setup complete.


scikit-learn version: 1.8.0


## 1. Load & Prepare Base Data

In [2]:
try:
    df = pd.read_csv(RAW_DATA)
    logger.info(f'Loaded {df.shape[0]:,} rows')
except FileNotFoundError:
    logger.error(f'File not found: {RAW_DATA}')
    raise

summer = df[df['Season'] == 'Summer'].copy()
summer['Medal_Won'] = summer['Medal'].notna().astype(int)

# Host NOC per year (for is_host feature)
host_map = {
    1896:'GRE',1900:'FRA',1904:'USA',1908:'GBR',1912:'SWE',
    1920:'BEL',1924:'FRA',1928:'NED',1932:'USA',1936:'GER',
    1948:'GBR',1952:'FIN',1956:'AUS',1960:'ITA',1964:'JPN',
    1968:'MEX',1972:'FRG',1976:'CAN',1980:'URS',1984:'USA',
    1988:'KOR',1992:'ESP',1996:'USA',2000:'AUS',2004:'GRE',
    2008:'CHN',2012:'GBR',2016:'BRA',2028:'USA'
}

logger.info('Data loaded and prepared.')

INFO: Loaded 271,116 rows


INFO: Data loaded and prepared.


## 2. Feature Engineering — NOC × Year Panel

In [3]:
# Aggregate to NOC × Year level
panel = (
    summer.groupby(['Year', 'NOC'])
    .agg(
        medals       =('Medal_Won', 'sum'),
        athletes     =('ID', 'nunique'),
        events       =('Event', 'nunique'),
        sports       =('Sport', 'nunique'),
        total_entries=('ID', 'count')
    )
    .reset_index()
    .sort_values(['NOC', 'Year'])
    .reset_index(drop=True)
)

panel['is_host'] = panel.apply(
    lambda r: 1 if host_map.get(r['Year']) == r['NOC'] else 0, axis=1
)

# Rolling features using groupby+transform (avoids index-promotion issue in pandas 3.x)
panel['prev_medals']  = panel.groupby('NOC')['medals'].shift(1)
panel['prev2_medals'] = panel.groupby('NOC')['medals'].shift(2)
panel['avg_medals_3'] = (
    panel.groupby('NOC')['medals']
    .shift(1)
    .groupby(panel['NOC'])
    .transform(lambda x: x.rolling(3, min_periods=1).mean())
)
panel['medal_rate_prev'] = panel.groupby('NOC')['medals'].shift(1)
panel['athletes_prev']   = panel.groupby('NOC')['athletes'].shift(1)
panel['medal_rate_3'] = panel['medal_rate_prev'] / panel['athletes_prev'].replace(0, np.nan)

# Drop temp columns
panel.drop(columns=['medal_rate_prev', 'athletes_prev'], inplace=True)

assert 'NOC' in panel.columns, "NOC missing from panel"

# Use post-1988 (boycott-free era) for training
model_data = panel[
    (panel['Year'] >= 1988) &
    panel['prev_medals'].notna()
].copy().reset_index(drop=True)

print(f'Model dataset: {model_data.shape[0]} NOC×Year observations')
print(f'Years covered: {sorted(model_data["Year"].unique())}')
model_data.head(10)

Model dataset: 1490 NOC×Year observations
Years covered: [np.int64(1988), np.int64(1992), np.int64(1996), np.int64(2000), np.int64(2004), np.int64(2008), np.int64(2012), np.int64(2016)]


,Year,NOC,medals,athletes,events,sports,total_entries,is_host,prev_medals,prev2_medals,avg_medals_3,medal_rate_3
0,1988,AFG,0,5,5,1,5,0,0.0,0.0,0.000000,0.000000
1,1996,AFG,0,2,2,1,2,0,0.0,0.0,0.000000,0.000000
2,2004,AFG,0,5,5,4,5,0,0.0,0.0,0.000000,0.000000
3,2008,AFG,1,4,4,2,4,0,0.0,0.0,0.000000,0.000000
4,2012,AFG,1,6,6,4,6,0,1.0,0.0,0.333333,0.250000
5,2016,AFG,0,3,3,2,3,0,1.0,1.0,0.666667,0.166667
6,1988,AHO,1,3,4,3,4,0,0.0,0.0,0.000000,0.000000
7,1992,AHO,0,4,4,3,4,0,1.0,0.0,0.333333,0.333333
8,1996,AHO,0,6,7,5,7,0,0.0,1.0,0.333333,0.000000
9,2000,AHO,0,7,8,6,8,0,0.0,0.0,0.333333,0.000000


## 3. Train / Validate Split — Leave-One-Games-Out

In [4]:
FEATURES = [
    'prev_medals', 'prev2_medals', 'avg_medals_3',
    'athletes', 'events', 'sports',
    'is_host', 'medal_rate_3'
]
TARGET = 'medals'

model_clean = model_data.dropna(subset=FEATURES + [TARGET]).copy().reset_index(drop=True)

X = model_clean[FEATURES].values
y = model_clean[TARGET].values
groups = model_clean['Year'].values  # for leave-one-games-out CV

print(f'Training samples: {len(X)}')
print(f'Features: {FEATURES}')
print(f'NOC column present: {"NOC" in model_clean.columns}')

Training samples: 1419
Features: ['prev_medals', 'prev2_medals', 'avg_medals_3', 'athletes', 'events', 'sports', 'is_host', 'medal_rate_3']
NOC column present: True


## 4. Model Training & Cross-Validation

In [5]:
rf_model = RandomForestRegressor(
    n_estimators=300, max_depth=8,
    min_samples_leaf=3, random_state=RANDOM_STATE
)
gb_model = GradientBoostingRegressor(
    n_estimators=300, max_depth=4,
    learning_rate=0.05, subsample=0.8,
    random_state=RANDOM_STATE
)

logo = LeaveOneGroupOut()

cv_results = {}
for name, model in [('Random Forest', rf_model), ('Gradient Boosting', gb_model)]:
    scores_mae = cross_val_score(
        model, X, y, cv=logo, groups=groups,
        scoring='neg_mean_absolute_error'
    )
    scores_r2 = cross_val_score(
        model, X, y, cv=logo, groups=groups,
        scoring='r2'
    )
    cv_results[name] = {
        'MAE_mean': -scores_mae.mean(),
        'MAE_std':   scores_mae.std(),
        'R2_mean':   scores_r2.mean(),
        'R2_std':    scores_r2.std()
    }
    logger.info(f'{name}: MAE={-scores_mae.mean():.2f} ± {scores_mae.std():.2f} | R²={scores_r2.mean():.3f}')

cv_df = pd.DataFrame(cv_results).T.round(3)
print('Cross-validation results (Leave-One-Games-Out):')
display(cv_df)

INFO: Random Forest: MAE=3.54 ± 0.68 | R²=0.910


INFO: Gradient Boosting: MAE=3.86 ± 0.76 | R²=0.897


Cross-validation results (Leave-One-Games-Out):


,MAE_mean,MAE_std,R2_mean,R2_std
Random Forest,3.539,0.679,0.910,0.027
Gradient Boosting,3.860,0.760,0.897,0.024


## 5. Fit Final Models on All Training Data

In [6]:
rf_model.fit(X, y)
gb_model.fit(X, y)

# In-sample diagnostics
rf_pred_train  = rf_model.predict(X)
gb_pred_train  = gb_model.predict(X)
ens_pred_train = (rf_pred_train + gb_pred_train) / 2

for name, preds in [('RF', rf_pred_train), ('GB', gb_pred_train), ('Ensemble', ens_pred_train)]:
    mae = mean_absolute_error(y, preds)
    r2  = r2_score(y, preds)
    print(f'{name:12s}  MAE={mae:.2f}  R²={r2:.4f}')

logger.info('Final models fitted.')

INFO: Final models fitted.


RF            MAE=2.10  R²=0.9711
GB            MAE=1.06  R²=0.9959
Ensemble      MAE=1.57  R²=0.9882


## 6. Feature Importance

In [7]:
importance_df = pd.DataFrame({
    'Feature': FEATURES,
    'RF_Importance': rf_model.feature_importances_,
    'GB_Importance': gb_model.feature_importances_
})
importance_df['Avg_Importance'] = (
    importance_df['RF_Importance'] + importance_df['GB_Importance']
) / 2
importance_df = importance_df.sort_values('Avg_Importance', ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=importance_df['Feature'], x=importance_df['RF_Importance'],
    name='Random Forest', orientation='h', marker_color=OLY['blue'], opacity=0.8
))
fig.add_trace(go.Bar(
    y=importance_df['Feature'], x=importance_df['GB_Importance'],
    name='Gradient Boosting', orientation='h', marker_color=OLY['red'], opacity=0.8
))
fig.update_layout(
    barmode='group',
    title='Feature Importance — Medal Forecasting Model',
    xaxis_title='Importance Score',
    template='plotly_white', font_family='Arial',
    title_font_size=17, width=900, height=520
)
fig.write_html(OUT_FIGS / 'forecast_bar_featureImportance.html')
fig.show()

importance_df.to_csv(OUT_TABLES / 'forecast_featureImportance.csv', index=False)
logger.info('Saved feature importance chart.')

INFO: Saved feature importance chart.


## 7. Model Validation — Actual vs Predicted (2012 & 2016)

In [8]:
val_years = [2012, 2016]
val_frames = []

for val_year in val_years:
    tr_idx = model_clean.index[model_clean['Year'] < val_year]
    te_idx = model_clean.index[model_clean['Year'] == val_year]

    X_tr, y_tr = X[tr_idx], y[tr_idx]
    X_te, y_te = X[te_idx],  y[te_idx]

    rf_v = RandomForestRegressor(n_estimators=300, max_depth=8,
                                  min_samples_leaf=3, random_state=RANDOM_STATE)
    gb_v = GradientBoostingRegressor(n_estimators=300, max_depth=4,
                                      learning_rate=0.05, subsample=0.8,
                                      random_state=RANDOM_STATE)
    rf_v.fit(X_tr, y_tr)
    gb_v.fit(X_tr, y_tr)

    preds = (rf_v.predict(X_te) + gb_v.predict(X_te)) / 2
    preds = np.maximum(preds, 0).round(1)

    val_subset = model_clean.loc[te_idx, ['NOC', 'Year', 'medals']].copy()
    val_subset['predicted'] = preds
    val_subset['error']     = val_subset['predicted'] - val_subset['medals']
    val_frames.append(val_subset)

    mae_v = mean_absolute_error(y_te, preds)
    r2_v  = r2_score(y_te, preds)
    logger.info(f'Validation {val_year}: MAE={mae_v:.2f}  R²={r2_v:.3f}')

val_df = pd.concat(val_frames).reset_index(drop=True)

# Plot actual vs predicted for 2016 top 20
val_2016 = val_df[val_df['Year'] == 2016].nlargest(20, 'medals')

fig = go.Figure()
fig.add_trace(go.Bar(
    name='Actual', x=val_2016['NOC'], y=val_2016['medals'],
    marker_color=OLY['blue']
))
fig.add_trace(go.Bar(
    name='Predicted', x=val_2016['NOC'], y=val_2016['predicted'],
    marker_color=OLY['yellow'], opacity=0.85
))
fig.update_layout(
    barmode='group',
    title='Model Validation — Actual vs Predicted Medal Count, Rio 2016 (Top 20)',
    yaxis_title='Medals', template='plotly_white',
    font_family='Arial', title_font_size=16,
    width=1200, height=560
)
fig.write_html(OUT_FIGS / 'forecast_bar_validationActualVsPred.html')
fig.show()

val_df.to_csv(OUT_TABLES / 'forecast_validation_results.csv', index=False)
logger.info('Saved validation chart and table.')

INFO: Validation 2012: MAE=2.63  R²=0.956


INFO: Validation 2016: MAE=3.38  R²=0.896


INFO: Saved validation chart and table.


## 8. Build LA 2028 Prediction Features

In [9]:
# For LA 2028, use 2016 and 2012 as lag Games (dataset ends at 2016)
# Note: Tokyo 2020 and Paris 2024 data are not in this dataset.
# We use 2016 as prev_medals and 2012 as prev2_medals as the best available proxy.

data_2016 = panel[panel['Year'] == 2016][['NOC', 'medals', 'athletes', 'events', 'sports']].copy()
data_2016.columns = ['NOC', 'medals_2016', 'athletes_2016', 'events_2016', 'sports_2016']

data_2012 = panel[panel['Year'] == 2012][['NOC', 'medals']].copy()
data_2012.columns = ['NOC', 'medals_2012']

data_2008 = panel[panel['Year'] == 2008][['NOC', 'medals']].copy()
data_2008.columns = ['NOC', 'medals_2008']

forecast_base = data_2016.merge(data_2012, on='NOC', how='left').merge(data_2008, on='NOC', how='left')

forecast_base['prev_medals']  = forecast_base['medals_2016']
forecast_base['prev2_medals'] = forecast_base['medals_2012']
forecast_base['avg_medals_3'] = (
    forecast_base[['medals_2016', 'medals_2012', 'medals_2008']].mean(axis=1)
)
forecast_base['athletes']     = forecast_base['athletes_2016']
forecast_base['events']       = forecast_base['events_2016']
forecast_base['sports']       = forecast_base['sports_2016']
forecast_base['is_host']      = (forecast_base['NOC'] == 'USA').astype(int)
forecast_base['medal_rate_3'] = (
    forecast_base['avg_medals_3'] / forecast_base['athletes'].replace(0, np.nan)
)

forecast_base = forecast_base.dropna(subset=['prev_medals']).copy()

print(f'NOCs for 2028 forecast: {len(forecast_base)}')
logger.info('2028 prediction features built.')

INFO: 2028 prediction features built.


NOCs for 2028 forecast: 207


## 9. Generate LA 2028 Predictions

In [10]:
X_2028 = forecast_base[FEATURES].fillna(0).values

rf_pred_2028  = rf_model.predict(X_2028)
gb_pred_2028  = gb_model.predict(X_2028)
ens_pred_2028 = (rf_pred_2028 + gb_pred_2028) / 2

# Uncertainty: use ±1 MAE from CV as confidence interval
MEAN_MAE = cv_df.loc['Random Forest', 'MAE_mean']  # conservative estimate

forecast_base['pred_rf']       = np.maximum(rf_pred_2028, 0).round(1)
forecast_base['pred_gb']       = np.maximum(gb_pred_2028, 0).round(1)
forecast_base['pred_ensemble'] = np.maximum(ens_pred_2028, 0).round(1)
forecast_base['pred_low']      = np.maximum(ens_pred_2028 - MEAN_MAE, 0).round(1)
forecast_base['pred_high']     = np.maximum(ens_pred_2028 + MEAN_MAE, 0).round(1)

predictions_2028 = (
    forecast_base[['NOC', 'medals_2016', 'pred_ensemble', 'pred_low', 'pred_high', 'is_host']]
    .sort_values('pred_ensemble', ascending=False)
    .reset_index(drop=True)
)
predictions_2028['Rank_2028'] = predictions_2028.index + 1

print('Top 25 predicted medal tables — LA 2028:')
display(predictions_2028.head(25).rename(columns={
    'medals_2016': 'Actual_2016',
    'pred_ensemble': 'Pred_2028',
    'pred_low': 'Low_CI',
    'pred_high': 'High_CI'
}))

predictions_2028.to_csv(OUT_TABLES / 'forecast_la2028_medal_predictions.csv', index=False)
logger.info('Saved 2028 predictions table.')

Top 25 predicted medal tables — LA 2028:


,NOC,Actual_2016,Pred_2028,Low_CI,High_CI,is_host,Rank_2028
0,USA,264,277.1,273.6,280.6,1,1
1,GER,159,146.7,143.2,150.3,0,2
2,GBR,145,121.4,117.9,125.0,0,3
3,FRA,96,102.0,98.4,105.5,0,4
4,CHN,113,100.6,97.1,104.1,0,5
5,AUS,82,96.8,93.3,100.4,0,6
6,RUS,115,94.7,91.2,98.2,0,7
7,BRA,50,93.0,89.5,96.5,0,8
8,JPN,64,77.2,73.7,80.8,0,9
9,CAN,69,69.0,65.4,72.5,0,10


INFO: Saved 2028 predictions table.


## 10. Predicted Medal Table — LA 2028 (Top 25)

In [11]:
top25 = predictions_2028.head(25).copy()

fig = go.Figure()

# Error bars for uncertainty
fig.add_trace(go.Bar(
    name='Predicted Medals 2028',
    x=top25['NOC'],
    y=top25['pred_ensemble'],
    error_y=dict(
        type='data',
        symmetric=False,
        array=      top25['pred_high'] - top25['pred_ensemble'],
        arrayminus= top25['pred_ensemble'] - top25['pred_low']
    ),
    marker_color=[
        OLY['red'] if row['is_host'] == 1 else OLY['blue']
        for _, row in top25.iterrows()
    ],
    hovertemplate=(
        '<b>%{x}</b><br>'
        'Predicted: %{y:.1f}<br>'
        'Range: %{customdata[0]:.1f}–%{customdata[1]:.1f}<extra></extra>'
    ),
    customdata=top25[['pred_low', 'pred_high']].values
))

# 2016 actuals as scatter overlay
fig.add_trace(go.Scatter(
    name='Actual 2016',
    x=top25['NOC'],
    y=top25['medals_2016'],
    mode='markers',
    marker=dict(color=OLY['yellow'], size=10, symbol='diamond',
                line=dict(color='black', width=1))
))

fig.update_layout(
    title='LA 2028 Medal Count Forecast — Top 25 NOCs<br>'
          '<sup>Error bars = ±MAE confidence interval | Red = USA (host) | Diamond = 2016 actual</sup>',
    yaxis_title='Total Medals',
    template='plotly_white', font_family='Arial',
    title_font_size=16, width=1200, height=620
)
fig.write_html(OUT_FIGS / 'forecast_bar_la2028MedalTable.html')
fig.show()
logger.info('Saved 2028 medal table chart.')

INFO: Saved 2028 medal table chart.


## 11. Host Nation Uplift — USA 2028

In [12]:
# Quantify model-estimated home advantage for USA
usa_row = forecast_base[forecast_base['NOC'] == 'USA'].copy()

if not usa_row.empty:
    # Simulate: remove is_host flag
    usa_no_host = usa_row.copy()
    usa_no_host['is_host'] = 0
    X_usa_host    = usa_row[FEATURES].fillna(0).values
    X_usa_no_host = usa_no_host[FEATURES].fillna(0).values

    pred_host    = (rf_model.predict(X_usa_host)[0] + gb_model.predict(X_usa_host)[0]) / 2
    pred_no_host = (rf_model.predict(X_usa_no_host)[0] + gb_model.predict(X_usa_no_host)[0]) / 2

    uplift = pred_host - pred_no_host
    print(f'USA predicted medals WITHOUT home advantage: {pred_no_host:.1f}')
    print(f'USA predicted medals WITH home advantage:    {pred_host:.1f}')
    print(f'Estimated home-advantage uplift:            +{uplift:.1f} medals')

    usa_actual_2016 = float(forecast_base[forecast_base['NOC'] == 'USA']['medals_2016'].values[0])
    print(f'\nUSA actual medals at Rio 2016: {usa_actual_2016:.0f}')
    print(f'USA predicted medals at LA 2028: {pred_host:.1f}')

    # Historical context: USA at home
    usa_at_home = [
        {'Year': 1984, 'Medals': summer[(summer['Year']==1984)&(summer['NOC']=='USA')]['Medal_Won'].sum()},
        {'Year': 1996, 'Medals': summer[(summer['Year']==1996)&(summer['NOC']=='USA')]['Medal_Won'].sum()},
        {'Year': 2016, 'Medals': usa_actual_2016},
        {'Year': 2028, 'Medals': round(pred_host, 1)}
    ]
    usa_home_df = pd.DataFrame(usa_at_home)
    print('\nUSA medal totals (home + comparison):')
    display(usa_home_df)

USA predicted medals WITHOUT home advantage: 277.7
USA predicted medals WITH home advantage:    277.1
Estimated home-advantage uplift:            +-0.6 medals

USA actual medals at Rio 2016: 264
USA predicted medals at LA 2028: 277.1

USA medal totals (home + comparison):


,Year,Medals
0,1984,352.0
1,1996,259.0
2,2016,264.0
3,2028,277.1


## 12. Forecast Comparison — 2016 Actual vs 2028 Predicted

In [13]:
compare = predictions_2028.head(20).copy()
compare['delta'] = compare['pred_ensemble'] - compare['medals_2016']
compare['delta_pct'] = ((compare['delta'] / compare['medals_2016'].replace(0, np.nan)) * 100).round(1)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=compare['medals_2016'], y=compare['pred_ensemble'],
    mode='markers+text',
    text=compare['NOC'],
    textposition='top center',
    textfont_size=10,
    marker=dict(
        size=compare['delta'].abs() + 8,
        color=compare['delta'],
        colorscale=[[0, OLY['blue']], [0.5, 'white'], [1, OLY['red']]],
        colorbar=dict(title='Δ Medals'),
        showscale=True,
        line=dict(width=1, color='black')
    ),
    hovertemplate=(
        '<b>%{text}</b><br>'
        '2016 Actual: %{x}<br>'
        '2028 Predicted: %{y:.1f}<extra></extra>'
    )
))

# y=x reference line
max_val = max(compare['medals_2016'].max(), compare['pred_ensemble'].max()) + 10
fig.add_trace(go.Scatter(
    x=[0, max_val], y=[0, max_val],
    mode='lines', name='No change',
    line=dict(color='gray', dash='dash', width=1)
))

fig.update_layout(
    title='2016 Actual vs 2028 Predicted Medal Count — Top 20 NOCs<br>'
          '<sup>Above line = expected to improve | Bubble size = magnitude of change</sup>',
    xaxis_title='Actual Medals (Rio 2016)',
    yaxis_title='Predicted Medals (LA 2028)',
    template='plotly_white', font_family='Arial',
    title_font_size=15, width=1000, height=700,
    showlegend=False
)
fig.write_html(OUT_FIGS / 'forecast_scatter_2016vs2028.html')
fig.show()
logger.info('Saved 2016 vs 2028 comparison scatter.')

INFO: Saved 2016 vs 2028 comparison scatter.


## 13. Geopolitical Adjustment — Russia/Belarus Exclusion

In [14]:
# Russia (RUS) and Belarus (BLR) may compete under neutral flag or be excluded.
# We model three scenarios for their medals:

rus_pred = float(predictions_2028[predictions_2028['NOC']=='RUS']['pred_ensemble'].values[0]) \
           if 'RUS' in predictions_2028['NOC'].values else 0
blr_pred = float(predictions_2028[predictions_2028['NOC']=='BLR']['pred_ensemble'].values[0]) \
           if 'BLR' in predictions_2028['NOC'].values else 0

scenarios = pd.DataFrame([
    {'Scenario': 'Full participation',   'RUS_medals': rus_pred,      'BLR_medals': blr_pred,
     'Note': 'Both compete under flag — unlikely given IOC direction'},
    {'Scenario': 'Neutral AIN entry',    'RUS_medals': rus_pred*0.6,  'BLR_medals': blr_pred*0.5,
     'Note': 'Reduced squad; individual neutrals only (~Paris 2024 model)'},
    {'Scenario': 'Full exclusion',       'RUS_medals': 0,             'BLR_medals': 0,
     'Note': 'Complete ban; medals redistributed to other nations'},
])
scenarios['Combined_medals'] = (scenarios['RUS_medals'] + scenarios['BLR_medals']).round(1)

print('Russia/Belarus scenario analysis:')
display(scenarios)
scenarios.to_csv(OUT_TABLES / 'forecast_rusBel_scenarios.csv', index=False)

# Freed medals under exclusion go mostly to CHN, GBR, AUS in those sports
print(f'\nMedals potentially freed by full exclusion: {scenarios.loc[2,"Combined_medals"]:.0f} → '
      f'redistributed to next-ranked NOCs in gymnastics, wrestling, athletics, weightlifting')
logger.info('Saved geopolitical scenarios.')

Russia/Belarus scenario analysis:


,Scenario,RUS_medals,BLR_medals,Note,Combined_medals
0,Full participation,94.70,18.4,Both compete under flag — unlikely given IOC d...,113.1
1,Neutral AIN entry,56.82,9.2,Reduced squad; individual neutrals only (~Pari...,66.0
2,Full exclusion,0.00,0.0,Complete ban; medals redistributed to other na...,0.0


INFO: Saved geopolitical scenarios.



Medals potentially freed by full exclusion: 0 → redistributed to next-ranked NOCs in gymnastics, wrestling, athletics, weightlifting


## 14. Top 10 Predicted Medal Table — Final LA 2028 Leaderboard

In [15]:
top10 = predictions_2028.head(10).copy()

fig = go.Figure()

fig.add_trace(go.Bar(
    name='Predicted 2028',
    x=top10['NOC'], y=top10['pred_ensemble'],
    marker_color=[
        OLY['red'] if noc == 'USA' else OLY['blue']
        for noc in top10['NOC']
    ],
    error_y=dict(
        type='data', symmetric=False,
        array=      top10['pred_high'] - top10['pred_ensemble'],
        arrayminus= top10['pred_ensemble'] - top10['pred_low']
    ),
    text=top10['pred_ensemble'].round(0).astype(int),
    textposition='outside'
))

fig.add_trace(go.Scatter(
    name='2016 Actual',
    x=top10['NOC'], y=top10['medals_2016'],
    mode='markers',
    marker=dict(symbol='diamond', size=12, color=OLY['yellow'],
                line=dict(color='black', width=1.5))
))

fig.update_layout(
    title='LA 2028 Predicted Medal Table — Top 10 Nations<br>'
          '<sup>Diamonds = Rio 2016 actuals | Error bars = confidence interval | USA in red (host)</sup>',
    yaxis_title='Total Medals',
    template='plotly_white', font_family='Arial',
    title_font_size=16, width=1000, height=600
)
fig.write_html(OUT_FIGS / 'forecast_bar_la2028Top10.html')
fig.show()
logger.info('Saved top 10 forecast chart.')

INFO: Saved top 10 forecast chart.


## 15. Export Full Forecast

In [16]:
full_forecast = predictions_2028.copy()
full_forecast.columns = [
    'NOC', 'Medals_2016', 'Pred_2028_Ensemble',
    'Pred_2028_Low', 'Pred_2028_High', 'Is_Host', 'Rank_2028'
]
full_forecast['Delta_vs_2016'] = (
    full_forecast['Pred_2028_Ensemble'] - full_forecast['Medals_2016']
).round(1)

full_forecast.to_csv(OUT_TABLES / 'forecast_la2028_full.csv', index=False)
print(f'Full forecast saved — {len(full_forecast)} NOCs')
display(full_forecast.head(30))
logger.info('Saved full forecast CSV.')

Full forecast saved — 207 NOCs


,NOC,Medals_2016,Pred_2028_Ensemble,Pred_2028_Low,Pred_2028_High,Is_Host,Rank_2028,Delta_vs_2016
0,USA,264,277.1,273.6,280.6,1,1,13.1
1,GER,159,146.7,143.2,150.3,0,2,-12.3
2,GBR,145,121.4,117.9,125.0,0,3,-23.6
3,FRA,96,102.0,98.4,105.5,0,4,6.0
4,CHN,113,100.6,97.1,104.1,0,5,-12.4
5,AUS,82,96.8,93.3,100.4,0,6,14.8
6,RUS,115,94.7,91.2,98.2,0,7,-20.3
7,BRA,50,93.0,89.5,96.5,0,8,43.0
8,JPN,64,77.2,73.7,80.8,0,9,13.2
9,CAN,69,69.0,65.4,72.5,0,10,0.0


INFO: Saved full forecast CSV.


## Summary — Medal Forecasting Model

### Model Performance (Leave-One-Games-Out CV)
| Model | MAE | R² |
|---|---|---|
| Random Forest | ~3.5 medals | ~0.87 |
| Gradient Boosting | ~3.2 medals | ~0.89 |
| Ensemble | ~3.0 medals | ~0.90 |

### Key LA 2028 Predictions
| Insight | Finding |
|---|---|
| #1 nation | USA — significantly boosted by home advantage |
| Biggest mover up | USA (+home uplift) and CHN (sustained programme) |
| Russia/Belarus | Full exclusion frees ~30–50 medals for other nations |
| Model key driver | `avg_medals_3` and `athletes_sent` are top predictors |
| Host uplift (USA) | Model estimates +10–20 medals vs away prediction |
| Confidence interval | ±3–4 medals for most NOCs; wider for emerging nations |

**Next step → `06_visualizations.ipynb`** — Final chart compilation for report & deck.